# Real-data analysis: distribution estimation + trajectories

Step 1 of the real-data pipeline, both halves in one place:

1. **Distribution estimation** -- the UOTReg barycenter at a dense grid of times, using ALL observed
   snapshots (no leave-out, unlike `05_realdata_loo`). Endpoints keep their raw observed cells.
2. **Trajectories** -- our UOT-map trajectory fitted on that anchored series, then clustered, scored
   against the WOT reference, and drawn.

Set `DATASET` to `"embryoid"` or `"statefate"` and run top to bottom; everything else that differs
between the two (the seed of record, the number of fate clusters, which day-0 cells are
transported) is set per dataset in the parameter cell.

**Nothing is written unless you set `SAVE = 1`**, and then it goes to `new_results/` -- the shipped
`results/` tree is never modified. With `SAVE = 0` (the default) the estimates and trajectories
stay in memory for the rest of the notebook.

In [ ]:
import os, sys, time, json
import numpy as np
import matplotlib.pyplot as plt

_here = os.path.abspath(os.path.dirname(__file__)) if "__file__" in globals() else os.getcwd()
# this folder (its helper modules) + tools/ (the shared `_repo.py` path resolver)
for _d in (_here, os.path.abspath(os.path.join(_here, os.pardir, os.pardir, "tools"))):
    if _d not in sys.path:
        sys.path.insert(0, _d)
import _analysis_common as A
import _repo as P
import uotreg as U
from uotreg.device import seed_everything
from uotreg.metrics import w2

# ----------------------------------------------------------------------------- SMOKE
# 1 = small and fast: runs end to end on a laptop. **NOT the paper's numbers.**
# 0 = the settings used in the paper.
SMOKE = globals().get("SMOKE", 1)
# 1 = write the estimates / trajectories / figures to `new_results/`; 0 = keep them in memory only.
SAVE = globals().get("SAVE", 0)
# The light artifacts (estimated clouds, trajectories, cluster labels, metrics) are a few MB.
# The heavy ones -- the trained generators, the fitted map chain, and the N x N distance matrix --
# are ~70 MB per dataset, and nothing in this repository needs them: the figures read the
# trajectory, and `cross_dimension` recomputes a missing distance matrix from it. Set
# SAVE_HEAVY = 1 only if you want to transport new cells later without refitting.
SAVE_HEAVY = 0

DATASET = globals().get("DATASET", "embryoid")    # "embryoid" | "statefate"
DIM     = globals().get("DIM", 20)               # 10 | 20 | 50 (d=50 reads the .h5ad)
DEVICE  = "cpu"           # "cuda" / "mps" for real runs

# Per-dataset settings of record. `seed` seeds the trajectory fit (network init + minibatch draws);
# `n_clust` is the published cluster count; `start` says which day-0 cells are transported --
# embryoid uses all of them, statefate the published 3000 (`starting3000.npy`), which are a
# SCATTERED subset of its 28249, not a prefix.
_DS = {"embryoid":  dict(seed=5, n_clust=6, start="all"),
       "statefate": dict(seed=2, n_clust=4, start="published")}[DATASET]
SEED       = _DS["seed"]
N_CLUST    = _DS["n_clust"]
START      = _DS["start"]                # "all" | "published" | "prefix"
N_START    = 3000                        # "published"/"prefix" only
CLUST_SEED = 0                           # the published clustering setting (KMeans/MDS random_state)
MDS_COMPONENTS = 10                      # published: n_components=10
TRAJ_TAU   = 50.0                        # trajectory maps use a LARGER tau than the estimator:
#                                          a small tau over-trims the one-sided map (overshoot)
VIZ_CELLS  = 1000                        # background cells drawn per cloud (None = all)
PUBLISHED_START = P.aux_path("data/scrna-statefate/starting3000.npy")

# reads resolve to your own `new_results/` run when you have one, else the shipped results
RESULTS  = A.results_dir(DATASET, DIM)
OUTDIR   = A.results_dir(DATASET, DIM, write=True) if SAVE else None
TUNE_DIR = (os.path.join(os.path.dirname(OUTDIR), f"d{DIM}_tune" + ("_smoke" if SMOKE else ""))
            if SAVE else None)

plt.rcParams.update({"font.size": 11, "axes.titlesize": 12, "axes.labelsize": 11,
                     "xtick.labelsize": 10, "ytick.labelsize": 10, "legend.fontsize": 9})

arrays, timepoints, labels = A.load_data(DATASET, DIM)
print(f"[{DATASET} d={DIM}] SMOKE={SMOKE} SAVE={SAVE} | observed times={timepoints}")
print(f"  reading estimates from {os.path.relpath(RESULTS, A.ROOT)}")
print(f"  writing to {os.path.relpath(TUNE_DIR, A.ROOT)}" if SAVE
      else "  SAVE=0 -- nothing will be written")

## Section 1 -- distribution estimation

The estimate at each interior grid time is an unbalanced-OT barycenter of the observed snapshots
under local-linear kernel weights. `A.est_recipe` returns the training recipe of record for this
dataset and dimension (d=50 gets a larger budget); edit `OVERRIDES` to change any knob -- the final
recipe is printed before anything runs.

In [ ]:
N_GEN    = 400 if SMOKE else 3000                      # points sampled per estimated cloud
SEED_EST = 0                                           # seeds the estimator (not the trajectory fit)

EST_T  = list(A.CFG[DATASET]["est_t"])                 # interior grid times to estimate (edit to add/remove)
TRAJ_T = A.CFG[DATASET]["traj_t"]
KW, USE_INI, INI = A.est_recipe(DATASET, DIM, smoke=SMOKE)   # <- the loo-matched (d-aware) defaults

# ---- EDIT HERE to change training params (applied on top of the recipe above) ----------------------
# Every estimate knob is a key in KW. Common ones (uncomment / edit to override):
OVERRIDES = {
    # "budget":   120,     # R = outer rounds (bigger = better-converged estimate; d=50 default 120)
    # "d_iters":  65,      # D (potential) inner iters per round     (d=50 default 65)
    # "t_iters":  12,      # T (map) inner iters per round           (d=50 default 12)
    # "g_iters":  65,      # G (generator) inner iters per round     (d=50 default 65)
    # "h":        4.0,     # local-Frechet bandwidth (embryoid 4.0 / statefate Silverman)
    # "tau":      5.0,     # UOT relaxation strength (embryoid 5 / statefate 1)
    # "gen_hidden": 256, "gen_layers": 4,   # generator width/depth
    # "map_hidden": 256, "map_layers": 5,   # map+potential width/depth
    # "init_iters": 15000, "gaussian_scale": 10.0,   # gaussian init
}
KW.update(OVERRIDES)
# ----------------------------------------------------------------------------------------------------

print(f"{DATASET} d={DIM} SMOKE={SMOKE} | observed times={timepoints}")
print(f"  trajectory grid={TRAJ_T}\n  estimate at interior={EST_T}  (endpoints use raw cells)  N_GEN={N_GEN}")
print(f"  use_ini={USE_INI}" + (f" ({INI})" if USE_INI else ""))
print("  FULL recipe (edit via OVERRIDES):")
for k in ("h", "tau", "budget", "d_iters", "t_iters", "g_iters", "gen_hidden", "gen_layers",
          "map_hidden", "map_layers", "batch_size", "batch_size_g", "init", "init_iters",
          "gaussian_scale", "divergence", "relaxation", "std_mode"):
    if k in KW:
        print(f"      {k:15s} = {KW[k]}")

### Estimate at each interior grid time
Each time is independent, so a crash costs only that time. With `SAVE = 1` the sampled cloud and
the trained generator are written per time; the generator lets you re-draw any number of points
later without re-estimating.

In [ ]:
EST_CLOUDS = {}          # time -> (n_gen, d) estimated cloud, kept in memory


def estimate_one(t):
    kw = dict(KW)
    if USE_INI:
        kw["pretrained_generator"] = INI
    cloud, est = U.estimate(arrays, timepoints, query_time=float(t), dim=DIM,
                            n_gen=N_GEN, seed=SEED_EST, return_estimator=True, **kw)
    EST_CLOUDS[float(t)] = np.asarray(cloud, np.float32)
    if SAVE:
        # `_smoke`-suffixed at SMOKE=1, and always under `new_results/` -- a run here can never
        # shadow or overwrite the shipped full-quality estimates.
        np.save(A.est_cloud_path(DATASET, DIM, t, smoke=SMOKE, write=True), EST_CLOUDS[float(t)])
        if SAVE_HEAVY:                                  # the trained generator (~3.6 MB per time)
            est.save(A.est_gen_path(DATASET, DIM, t, smoke=SMOKE, write=True))
    return cloud


for t in EST_T:
    t0 = time.time()
    c = estimate_one(t)
    print(f"  [Day{t}] estimated {c.shape} in {time.time()-t0:.0f}s"
          + (" -> saved cloud + generator" if SAVE else ""))

### The estimated series
Raw observed cells at the two endpoints, the estimate at every interior grid time. This is exactly
the series the trajectory is fitted on below. Estimates made in this session are used when present;
otherwise the shipped ones are loaded, so Section 2 runs whether or not you ran Section 1.

In [ ]:
def analysis_series():
    """The anchored series over the trajectory grid: raw endpoints, estimated interior.

    An estimate made in THIS session wins; otherwise the saved one is loaded (yours from
    `new_results/` if you have it, else the shipped one)."""
    out = []
    for i, t in enumerate(TRAJ_T):
        if i == 0 or i == len(TRAJ_T) - 1:
            out.append(arrays[list(timepoints).index(float(t))])
        elif float(t) in EST_CLOUDS:
            out.append(EST_CLOUDS[float(t)])
        else:
            fp = A.est_cloud_path(DATASET, DIM, t)
            if not os.path.exists(fp):
                raise FileNotFoundError(f"missing estimate {fp} -- run Section 1 for {DATASET} d={DIM}")
            out.append(np.load(fp))
    return out, [float(t) for t in TRAJ_T]


series, grid = analysis_series()
fig, ax = plt.subplots(figsize=(6.4, 5.2), dpi=140)
A.plot_estimated_distributions(series, grid, ax=ax,
    title=f"{DATASET} d={DIM}: estimated distributions over time")
ax.legend(fontsize=8, ncol=2, loc="best")
fig.tight_layout()
if SAVE:
    os.makedirs(os.path.join(OUTDIR, "figs"), exist_ok=True)
    for ext in ("pdf", "png"):
        fig.savefig(os.path.join(OUTDIR, "figs", f"est_series_{DATASET}{DIM}.{ext}"),
                    bbox_inches="tight", dpi=200)
plt.show()

## Section 2 -- trajectories

The cells transported are the day-0 cells of record: all of them for embryoid, the published 3000
for statefate. The published statefate set is stored as coordinates, so its row indices are
recovered by an exact match against the day-0 cloud and saved with the run -- without them the
cluster labels cannot be joined back to the gene-space rows.

In [ ]:
def _published_start(arrays0, dedupe=False):
    """(cells, indices) for the paper's 3000 -- matched back to rows of the current day-0 cloud.

    The npy stores COORDINATES, not indices, so the analysis's row order is recoverable only by
    matching values. Matching is bit-exact on float32 and finds all 3000, so these ARE the published
    cells, not an approximation.

    ⚠ **The published file holds 3000 draws but only 2850 DISTINCT cells** -- 150 appear twice. The
    day-0 cloud itself has no duplicate rows (28249/28249 unique), so this is not a matching artifact:
    the paper drew 3000 cells WITH REPLACEMENT. Keeping the duplicates reproduces the published
    analysis exactly (a repeated cell contributes two identical trajectories and is counted twice by
    KMeans); `dedupe=True` drops them for a cleaner 2850-cell run that is NOT what was published."""
    # The stored coordinates are 20-dimensional, so the match MUST be made against the d=20 day-0
    # cloud -- at another DIM the cloud has a different width and nothing matches. Resolving the
    # indices at d=20 and reusing them is exact: the PC bases are nested and the row order is
    # identical across d, so row i is the same cell at every dimension.
    S = np.ascontiguousarray(np.load(PUBLISHED_START), np.float32)
    ref0 = np.asarray(A.load_data(DATASET, S.shape[1])[0][0], np.float32)
    assert len(ref0) == len(arrays0), (
        f"day-0 cloud has {len(arrays0)} cells at d={DIM} but {len(ref0)} at d={S.shape[1]} -- the "
        f"row correspondence these indices rely on does not hold")
    key = {r.tobytes(): i for i, r in enumerate(np.ascontiguousarray(ref0, np.float32))}
    idx = np.array([key.get(r.tobytes(), -1) for r in S])
    missing = int((idx < 0).sum())
    if missing:
        raise RuntimeError(f"{missing}/{len(S)} published start cells not found in the d={S.shape[1]} "
                           f"day-0 cloud -- the loader or the PCA changed; use RA_SF_START=prefix instead")
    n_dup = len(idx) - len(set(idx.tolist()))
    if dedupe:
        _, first = np.unique(idx, return_index=True)
        keep = np.sort(first)
        print(f"  [start] published set: {len(S)} draws -> {len(keep)} distinct cells (dropped {n_dup} "
              f"duplicates -- this is NOT the published analysis)")
        return np.asarray(arrays0, np.float32)[idx[keep]], idx[keep]
    print(f"  [start] published set: {len(S)} draws, {len(set(idx.tolist()))} distinct "
          f"({n_dup} cells drawn twice -- kept, as published) | d={DIM} coordinates")
    # the CELLS are taken from the current dimension's cloud, not from the 20-d npy: at d=20 this is
    # bit-identical to returning `S`, and at d=10/50 it is the only correct answer.
    return np.asarray(arrays0, np.float32)[idx], idx


n0 = len(arrays[0])
if START == "published":
    X0, X0_IDX = _published_start(arrays[0], dedupe=False)
elif START == "prefix":
    X0, X0_IDX = arrays[0][:min(N_START, n0)], np.arange(min(N_START, n0))
else:                                                     # "all" -- the published embryoid setting
    X0, X0_IDX = arrays[0], np.arange(n0)
N_ALL = N_CLUST_CELLS = len(X0)
print(f"  start cells {N_ALL}/{n0} | clustering all of them into {N_CLUST} clusters "
      f"| tau={TRAJ_TAU} | device={DEVICE}")

RUNS = {}          # seed -> everything about one fitted run (in memory until saved)

### The WOT reference (fit once, reused by every run)

In [ ]:
WOT = {}


def wot_reference(force=False):
    """Fit (or reload) the WOT comparison trajectory on the observed snapshots."""
    if WOT and not force:
        return WOT
    t0 = time.time()
    saved = A.model_path(DATASET, DIM, "WOT")
    if os.path.exists(saved) and not force:
        tf = A.load_uot_model(DATASET, DIM, "WOT", DIM, device=DEVICE)
        tr = np.asarray(tf.transport(X0), np.float32)
        print(f"  WOT reloaded from the saved d20 model {tr.shape} ({time.time()-t0:.0f}s)")
    else:
        seed_everything(0)
        tr = A.fit_ours_uot(arrays, timepoints, X0, relaxation="balanced", dim=DIM,
                            device=DEVICE, dataset=DATASET)
        print(f"  WOT fitted on the observed snapshots {tr.shape} ({time.time()-t0:.0f}s)")
    WOT.update(traj=np.asarray(tr, np.float32), grid=[float(t) for t in timepoints])
    return WOT


wot_reference()

### Metrics

In [ ]:
def _spread(cloud):
    c = np.asarray(cloud, float)
    return float(np.linalg.norm(c - c.mean(0), axis=1).mean())


def scored(tr, grid_, n_sub=1000, seed=0):
    """A.traj_metrics + the scale-aware companions."""
    m = A.traj_metrics(tr, grid_, arrays, timepoints, n_sub=n_sub, seed=seed)
    trv, tv = A.obs_time_view(tr, grid_, timepoints)          # observed-times view -> fair comparison
    # spread ratio: transported cloud spread / observed spread, averaged over the observed times
    ratios = []
    for i, t in enumerate(tv):
        obs = arrays[[float(x) for x in timepoints].index(float(t))]
        ratios.append(_spread(trv[i, :n_sub]) / max(_spread(obs), 1e-9))
    m["spread"] = float(np.mean(ratios))
    # purity normalized by the trajectory's own scale (purity is an absolute distance -> it rewards
    # contraction; dividing by the mean pairwise distance removes that advantage)
    sub = trv[:, :min(n_sub, trv.shape[1]), :]
    DM = A.pairwise_aligned_euclid(sub)
    m["purity_norm"] = float(m["purity_obs"] / max(DM[np.triu_indices_from(DM, 1)].mean(), 1e-9))
    return m


def metric_table(runs=None):
    runs = runs or RUNS
    if not runs:
        print("no runs yet"); return
    w = wot_reference()
    if "metrics" not in w:
        w["metrics"] = scored(w["traj"], w["grid"])
    keys = ["fidelity", "purity_obs", "purity_norm", "spread", "retention"]
    print(f"  {'run':22s}" + "".join(f"{k:>13}" for k in keys))
    for k in sorted(runs):
        m = runs[k]["metrics"]
        print(f"  {'ours seed ' + str(k):22s}" + "".join(f"{m[x]:13.3f}" for x in keys))
    print(f"  {'WOT (observed grid)':22s}" + "".join(f"{w['metrics'][x]:13.3f}" for x in keys))
    print("\n  fidelity   = mean W2(traj, observed snapshot) at the observed times   LOWER is better")
    print("  purity_obs = neighbourhood purity, observed-times view (grid-matched)  LOWER is tighter")
    print("  purity_norm= purity_obs / mean pairwise distance -- the SCALE-FREE version.")
    print("               Raw purity rewards a contracted cloud, and ours contracts (see `spread`),")
    print("               so quote purity_norm, or quote fidelity + spread instead.")
    print("  spread     = traj spread / observed spread at the observed times       ~1 is faithful")

### Fit

In [ ]:
def fit(seed=None, tau=None, d_iters=None, t_iters=None, clust_seed=None, n_clust=None, viz=True):
    """Fit OUR UOT-map trajectory on the saved d=20 estimates, cluster it, score it, draw it."""
    seed = SEED if seed is None else int(seed)
    cs = clust_seed if clust_seed is not None else (CLUST_SEED if CLUST_SEED is not None else seed)
    nc = n_clust or N_CLUST
    t0 = time.time()
    seed_everything(seed)                       # <- the one call that makes the FIT reproducible
    # SMOKE shortens the two inner loops; everything else (the data, the estimates, the
    # clustering) is unchanged, so the plumbing is exercised without the cost.
    if SMOKE:
        d_iters = 20 if d_iters is None else d_iters
        t_iters = 10 if t_iters is None else t_iters
    traj, tf = A.fit_ours_uot(series, grid, X0, relaxation="one-sided", dim=DIM, device=DEVICE,
                              dataset=DATASET, tau=(TRAJ_TAU if tau is None else float(tau)),
                              d_iters=d_iters, t_iters=t_iters, return_model=True)
    traj = np.asarray(traj, np.float32)
    sub = traj[:, :min(N_CLUST_CELLS, traj.shape[1]), :]           # = all of them, as published
    DM = A.pairwise_aligned_euclid(sub)
    labels, _ = A.kmeans_from_dist(DM, n_clusters=nc, n_components=MDS_COMPONENTS,
                                   random_state=cs)                  # <- clustering reproducible
    m = scored(traj, grid)
    RUNS[seed] = dict(seed=seed, clust_seed=cs, n_clust=nc, traj=traj, model=tf, DM=DM,
                      labels=labels, metrics=m, grid=[float(t) for t in grid],
                      cfg=dict(tau=(TRAJ_TAU if tau is None else float(tau)), d_iters=d_iters,
                               t_iters=t_iters, n_clust=nc, clust_seed=cs, n_all=int(N_ALL),
                               start=START, mds_components=MDS_COMPONENTS,
                               n_clust_cells=int(N_CLUST_CELLS)),
                      seconds=time.time() - t0)
    print(f"[seed {seed}] fitted {traj.shape} in {time.time()-t0:.0f}s | "
          f"fidelity {m['fidelity']:.3f}  purity_obs {m['purity_obs']:.3f}  "
          f"purity_norm {m['purity_norm']:.3f}  spread {m['spread']:.3f}")
    if viz:
        show(seed)
    return RUNS[seed]

### Look at it

In [ ]:
def show(seed=None, n_paths=14, path_seed=0, save_to=None):
    seed = SEED if seed is None else int(seed)
    r = RUNS[seed]; w = wot_reference()
    if "metrics" not in w:
        w["metrics"] = scored(w["traj"], w["grid"])
    # main row: input distributions | our paths | per-time fidelity vs WOT
    fig, (ax1, ax2, ax3) = plt.subplots(1, 3, figsize=(16, 4.6), dpi=130)
    A.plot_estimated_distributions_color(series, grid, ax=ax1, max_cells=VIZ_CELLS,
                                         title=f"{DATASET} d={DIM}: estimated distributions (fitted input)")
    A.plot_trajectory_color(series, grid, r["traj"], n_paths=n_paths, ax=ax2, seed=path_seed,
                            max_cells=VIZ_CELLS,
                            title=f"ours: UOT maps — seed {seed}")
    fo = r["metrics"]["fidelity_t"]; fw = w["metrics"]["fidelity_t"]
    ts = sorted(fo); ax3.plot(ts, [fo[t] for t in ts], "o-", label=f"ours (seed {seed})")
    tw = sorted(fw); ax3.plot(tw, [fw[t] for t in tw], "s--", c="0.5", label="WOT")
    ax3.set_xlabel("day"); ax3.set_ylabel("W2 to the observed snapshot")
    ax3.set_title("fidelity per observed time (lower = better)"); ax3.legend()
    fig.tight_layout()
    if save_to:
        fig.savefig(save_to, bbox_inches="tight", dpi=200); print("saved", save_to)
    plt.show()
    # clusters: `plot_clusters_medoids` draws its OWN figure (no ax argument)
    figc = A.plot_clusters_medoids(r["traj"][:, :r["DM"].shape[0]], r["labels"], r["DM"],
                                   title=f"{DATASET} d={DIM} seed {seed}: clusters "
                                         f"(k={r['n_clust']}, clust_seed={r['clust_seed']})", seed=path_seed)
    if save_to:
        f2 = save_to.replace(".png", "_clusters.png")
        (figc or plt.gcf()).savefig(f2, bbox_inches="tight", dpi=200); print("saved", f2)
    plt.show()
    metric_table({seed: r})
    return fig

### Save the run
Writes the trajectory, the distance matrix, the cluster labels, the day-0 row indices, the trained
map chain and a metadata json into `new_results/realdata_analysis/<ds>/d<DIM>_tune/seed<k>/` --
the layout the fate, figure and cross-dimension notebooks read. Only runs when `SAVE = 1`.

In [ ]:
def save_run(seed=None, overwrite=False, fig=True):
    seed = SEED if seed is None else int(seed)
    assert seed in RUNS, f"no run for seed {seed} -- call fit({seed}) first"
    r = RUNS[seed]
    d = os.path.join(TUNE_DIR, f"seed{seed}")
    if os.path.exists(d) and not overwrite and os.listdir(d):
        raise FileExistsError(f"{d} already has files -- pass overwrite=True to replace them")
    os.makedirs(d, exist_ok=True)
    tag = f"{DATASET}{DIM}_oursUOTmaps"
    np.save(os.path.join(d, f"traj_{tag}.npy"), r["traj"])
    if SAVE_HEAVY:      # N x N, ~23 MB at 2381 cells -- cross_dimension recomputes it if absent
        np.save(os.path.join(d, f"dm_{tag}.npy"), r["DM"].astype(np.float32))
    np.save(os.path.join(d, f"clusters_{tag}.npy"), r["labels"])
    # WHICH day-0 cells these are. The published set is scattered, so without this the labels cannot
    # be joined back to the h5ad rows (gene-space DE needs exactly that join).
    np.save(os.path.join(d, f"startidx_{tag}.npy"), X0_IDX)
    if SAVE_HEAVY:      # ~20 MB; reload with TrajectoryFitter.load to transport other cells later
        r["model"].save(os.path.join(d, f"model_{tag}.pth"))
    A.save_json(dict(dataset=DATASET, dim=DIM, seed=seed, grid=r["grid"], cfg=r["cfg"],
                     start=START, n_start=int(N_ALL), start_source=(PUBLISHED_START
                                                                    if START == "published" else "prefix"),
                     metrics={k: v for k, v in r["metrics"].items() if k != "fidelity_t"},
                     fidelity_t={str(k): v for k, v in r["metrics"]["fidelity_t"].items()},
                     wot_metrics={k: v for k, v in wot_reference()["metrics"].items() if k != "fidelity_t"},
                     seconds=r["seconds"]),
                os.path.join(d, f"meta_{tag}.json"))
    if fig:
        show(seed, save_to=os.path.join(d, f"fig_{tag}.png"))
    print(f"[seed {seed}] saved -> {d}")
    return d


def list_saved():
    if not os.path.isdir(TUNE_DIR):
        print("nothing saved yet"); return
    for s in sorted(os.listdir(TUNE_DIR)):
        p = os.path.join(TUNE_DIR, s)
        if not os.path.isdir(p):
            continue
        mf = [f for f in os.listdir(p) if f.startswith("meta_")]
        if mf:
            j = json.load(open(os.path.join(p, mf[0])))
            m = j["metrics"]
            print(f"  {s:10s} fidelity {m['fidelity']:.3f}  purity_obs {m['purity_obs']:.3f}  "
                  f"purity_norm {m['purity_norm']:.3f}  spread {m['spread']:.3f}  cfg={j['cfg']}")
        else:
            print(f"  {s:10s} (no meta json)")

### Run it

In [ ]:
fit(SEED)

if SAVE:
    save_run(SEED, overwrite=True)
else:
    print("SAVE=0 -- run kept in memory only (set SAVE=1 to write it to new_results/)")